# DemandIQ: Model Experimentation, Cross-Validation & Self-Improving Loop 🧪

**Project:** DemandIQ — AI Demand Forecasting & Inventory Intelligence  
**Objective:** End-to-end model evaluation, XGBoost training with early stopping, time-series cross-validation backtesting, SHAP driver attribution, and active self-improving loop testing.

In [ ]:
import pandas as pd

from demandiq.config import DEFAULT_CONFIG, ModelConfig
from demandiq.features import build_feature_pipeline
from demandiq.harness.backtesting import TimeSeriesBacktester
from demandiq.harness.stress_testing import ScenarioStressTester
from demandiq.loop.self_improving import AdaptiveSelfImprovingLoop
from demandiq.models import XGBoostDemandForecaster
from demandiq.pipeline import generate_synthetic_sample_data, run_training_pipeline

print("DemandIQ experimentation modules loaded.")

## 1. Execute End-to-End Pipeline & Benchmark Baseline vs. XGBoost

In [ ]:
results = run_training_pipeline(
    config=DEFAULT_CONFIG, use_synthetic_fallback=True, save_artifacts=True
)

print("\n=== Benchmark Comparison ===")
print("Baseline Forecaster:", results["baseline_metrics"])
print("XGBoost Forecaster :", results["xgboost_metrics"])
print("Inventory KPIs     :", results["inventory_kpis"])

## 2. Multi-Horizon Time-Series Cross-Validation Harness
Run expanding-window backtesting across historical splits to evaluate stability across operational slices.

In [ ]:
sample_data = generate_synthetic_sample_data(num_weeks=145, num_centers=5, num_meals=8)
raw_merged = (
    sample_data["train"]
    .merge(sample_data["centers"], on="center_id")
    .merge(sample_data["meals"], on="meal_id")
)
df_proc, feat_cols, _ = build_feature_pipeline(raw_merged)

backtester = TimeSeriesBacktester(n_folds=3, test_window_weeks=10, min_train_weeks=50)


def model_factory():
    return XGBoostDemandForecaster(model_config=ModelConfig(n_estimators=100, max_depth=5))


backtest_result = backtester.run_backtest(df_proc, model_factory, feat_cols)
print("Overall Backtest Metrics:", backtest_result.overall_metrics)

fold_summary = pd.DataFrame([f.__dict__ for f in backtest_result.fold_evaluations])
fold_summary

## 3. Operational Stress Testing & Disruption Simulation
Simulating price shock, marketing surge, and blackout scenarios.

In [ ]:
model = XGBoostDemandForecaster(model_config=ModelConfig(n_estimators=100, max_depth=5))
model.fit(df_proc[feat_cols], df_proc["num_orders"])

stress_tester = ScenarioStressTester(model, feat_cols)
stress_res = stress_tester.run_all_stress_tests(df_proc)
print("Stress Test Verdict:", stress_res.summary_verdict)
pd.DataFrame([o.__dict__ for o in stress_res.scenario_outcomes])

## 4. Hyper-Personalized Self-Improving Loop Demonstration
Demonstrating active online bias correction and Champion-Challenger evaluation.

In [ ]:
loop_engine = AdaptiveSelfImprovingLoop(champion_model=model, feature_cols=feat_cols)
loop_res = loop_engine.process_incoming_batch(df_proc.iloc[-400:], actual_col="num_orders")

print(f"Champion Base WMAPE : {loop_res.champion_wmape:.4f}")
print(f"Adaptive Loop WMAPE : {loop_res.adaptive_wmape:.4f}")
print(f"Improvement Gain    : {loop_res.bias_correction_improvement_pct:.2f}%")
print(f"Promoted Challenger : {loop_res.promoted_new_champion}")